In [2]:
!pip -q install simpy ipywidgets pandas matplotlib numpy

import simpy
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from collections import defaultdict
from google.colab import files

PROJECT_NAME = "Bank Queue & Customer Service Simulation"

DEFAULT_DURATION = 480
DEFAULT_TELLERS = 3
DEFAULT_ARRIVAL = 3
DEFAULT_SERVICE = 5
DEFAULT_SEED = 42

class BankSimulation:

    def __init__(
        self,
        duration=480,
        tellers=3,
        arrival=3,
        service=5,
        seed=42
    ):

        self.duration = duration
        self.teller_count = tellers
        self.arrival = arrival
        self.service = service

        self.rng = random.Random(seed)
        self.env = simpy.Environment()

        self.available_tellers = simpy.Store(
            self.env,
            capacity=tellers
        )

        for i in range(1, tellers + 1):
            self.available_tellers.put(i)

        self.total_customers = 0
        self.waiting_count = 0

        self.records = []

        self.teller_busy_time = defaultdict(float)
        self.busy_tellers = set()

        self.queue_history = []
        self.busy_history = []

        self.max_queue = 0

    def customer(self, customer_id):

        arrival_time = self.env.now

        self.waiting_count += 1

        self.max_queue = max(
            self.max_queue,
            self.waiting_count
        )

        teller_id = yield self.available_tellers.get()

        self.waiting_count -= 1

        service_start = self.env.now

        waiting_time = service_start - arrival_time

        service_duration = self.rng.expovariate(
            1 / self.service
        )

        self.busy_tellers.add(teller_id)

        yield self.env.timeout(service_duration)

        departure_time = self.env.now

        self.busy_tellers.remove(teller_id)

        actual_busy_time = max(
            0,
            min(departure_time, self.duration)
            - min(service_start, self.duration)
        )

        self.teller_busy_time[teller_id] += actual_busy_time

        self.records.append({

            "Customer ID": customer_id,
            "Teller": teller_id,
            "Arrival Time": arrival_time,
            "Service Start": service_start,
            "Service Time": service_duration,
            "Waiting Time": waiting_time,
            "Departure Time": departure_time,
            "Time in System": departure_time - arrival_time

        })

        yield self.available_tellers.put(teller_id)

    def arrival_process(self):

        customer_id = 0

        while True:

            interval = self.rng.expovariate(
                1 / self.arrival
            )

            yield self.env.timeout(interval)

            if self.env.now >= self.duration:
                break

            customer_id += 1

            self.total_customers += 1

            self.env.process(
                self.customer(customer_id)
            )

    def monitor(self):

        while self.env.now < self.duration:

            self.queue_history.append({

                "Time": self.env.now,
                "Queue Length": self.waiting_count

            })

            self.busy_history.append({

                "Time": self.env.now,
                "Busy Tellers": len(self.busy_tellers)

            })

            yield self.env.timeout(1)

    def run(self):

        self.env.process(self.arrival_process())

        self.env.process(self.monitor())

        self.env.run(until=self.duration)

        self.env.run()

        self.records.sort(
            key=lambda x: x["Customer ID"]
        )

        return self

    def get_results(self):

        df = pd.DataFrame(self.records)

        if df.empty:
            return None

        total_busy = sum(
            self.teller_busy_time.values()
        )

        utilization = (
            total_busy /
            (self.teller_count * self.duration)
        ) * 100

        return {

            "Total Customers": self.total_customers,

            "Served Customers": len(df),

            "Average Waiting Time":
                df["Waiting Time"].mean(),

            "Average Service Time":
                df["Service Time"].mean(),

            "Average Time in System":
                df["Time in System"].mean(),

            "Teller Utilization": utilization,

            "Maximum Queue Length": self.max_queue,

            "Customer Records": df

        }

display(HTML("""
<div style="
    background:linear-gradient(120deg,#101827,#17365d);
    padding:28px;
    border-radius:15px;
    color:white;
    text-align:center;
    margin-bottom:20px;
">
    <h1 style="color:#60a5fa;">
        🏦 BANK QUEUE SIMULATION
    </h1>

    <h3>Customer Service Management System</h3>

    <p>Simulation and Modeling Lab Project</p>

    <p>Python | SimPy | Pandas | Matplotlib</p>
</div>
"""))

teller_input = widgets.IntSlider(
    value=3,
    min=1,
    max=10,
    description="Tellers:",
    style={"description_width":"120px"}
)

arrival_input = widgets.IntSlider(
    value=3,
    min=1,
    max=20,
    description="Arrival Interval:",
    style={"description_width":"120px"}
)

service_input = widgets.IntSlider(
    value=5,
    min=1,
    max=30,
    description="Service Time:",
    style={"description_width":"120px"}
)

duration_input = widgets.IntSlider(
    value=480,
    min=60,
    max=720,
    step=60,
    description="Duration:",
    style={"description_width":"120px"}
)

seed_input = widgets.IntText(
    value=42,
    description="Random Seed:",
    style={"description_width":"120px"}
)

start_button = widgets.Button(
    description="START SIMULATION",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="190px")
)

reset_button = widgets.Button(
    description="RESET",
    button_style="danger",
    icon="refresh",
    layout=widgets.Layout(width="120px")
)

download_button = widgets.Button(
    description="DOWNLOAD CSV",
    button_style="info",
    icon="download",
    layout=widgets.Layout(width="170px")
)

output = widgets.Output()

current_simulation = {
    "sim": None,
    "comparison": None
}

def show_kpi(data):

    cards = [

        ("Total Customers",
         data["Total Customers"]),

        ("Served Customers",
         data["Served Customers"]),

        ("Average Waiting",
         f'{data["Average Waiting Time"]:.2f} min'),

        ("Average Service",
         f'{data["Average Service Time"]:.2f} min'),

        ("Teller Utilization",
         f'{data["Teller Utilization"]:.2f}%'),

        ("Maximum Queue",
         data["Maximum Queue Length"])

    ]

    html = """
    <div style="
        display:grid;
        grid-template-columns:repeat(3,minmax(0,1fr));
        gap:12px;
        margin:20px 0;
    ">
    """

    for title, value in cards:

        html += f"""
        <div style="
            background:#172338


BANK QUEUE SIMULATION SYSTEM READY!
Configure parameters and click START SIMULATION.
